# AttGAN — Facial Attribute Editing with GANs
**Dataset:** CelebA · **Framework:** PyTorch · **Platform:** Google Colab

> **Paper:** He et al. (2019) — *AttGAN: Facial Attribute Editing by Only Changing What You Want*

---
### How to run
1. `Runtime → Change runtime type → GPU (T4)`
2. Run cells in order (Shift+Enter)
3. CelebA downloads automatically on first run (~1.4 GB)

All source code lives in `src/`. Configuration is in `config.py`.

## Cell 1 — Clone repo & install dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/YOUR_USERNAME/GAN_Project_DL.git
%cd GAN_Project_DL

# Install requirements
!pip install -q -r requirements.txt

# Verify GPU
import torch, sys
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('⚠️  No GPU — go to Runtime → Change Runtime Type → GPU')

## Cell 2 — Configuration

> Edit `config.py` to change hyperparameters. No code changes needed here.

In [ ]:
import torch
from config import Config

cfg    = Config()          # creates data/ results/ checkpoints/ if missing
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Device     : {device}')
print(f'Image size : {cfg.IMG_SIZE}×{cfg.IMG_SIZE}')
print(f'Batch size : {cfg.BATCH_SIZE}')
print(f'Epochs     : {cfg.N_EPOCHS}')
print(f'Attributes : {cfg.ATTRS}')
print(f'λ_rec={cfg.LAMBDA_REC}  λ_cls_D={cfg.LAMBDA_CLS_D}  λ_cls_G={cfg.LAMBDA_CLS_G}')

## Cell 3 — Load CelebA dataset

**Key concept:** CelebA has 40 binary attribute labels per image.
We select 13 and convert {0, 1} → {−1, +1} for bipolar conditioning.

> First run downloads ~1.4 GB. Subsequent runs use the local cache.
> If download fails (Google Drive quota), see the troubleshooting cell at the bottom.

In [ ]:
from src.dataset import get_loaders

train_loader, test_loader = get_loaders(cfg)

# Sanity check
imgs, attrs = next(iter(train_loader))
print(f'Image batch : {imgs.shape}   (B, C, H, W)')
print(f'Attr  batch : {attrs.shape}  values in {set(attrs.unique().tolist())}')

## Cell 4 — Build models

**AttGAN architecture:**
```
image ──► Encoder ──► z (latent)
                       │
          attrs_target ──► concat
                               │
                           Generator ──► fake_img
                                              │
                          Discriminator ◄─────┘
                               ├─► adv_head  (real/fake)
                               └─► cls_head  (13 attr logits)
```

In [ ]:
from src.models import build_models

enc, gen, dis = build_models(cfg, device)

## Cell 5 — Train

**Three losses:**

| Loss | Formula | Purpose |
|------|---------|--------|
| `L_adv` | MSELoss (LSGAN) | Makes fakes look real |
| `L_cls` | BCEWithLogitsLoss | Correct attributes appear |
| `L_rec` | L1Loss · λ=100 | Preserves identity |

**Total G loss:** `L_adv + λ_cls_G·L_cls + λ_rec·L_rec`

In [ ]:
from src.trainer import Trainer

trainer = Trainer(enc, gen, dis, train_loader, test_loader, cfg, device)

# To resume from a checkpoint, pass:
#   trainer.train(resume_path='checkpoints/ckpt_epoch010.pt')
g_losses, d_losses = trainer.train()

## Cell 6 — Loss curves

In [ ]:
from src.utils import plot_losses
plot_losses(g_losses, d_losses, cfg)

## Cell 7 — Attribute manipulation demo

For each test image, every attribute is toggled independently.
This is the main qualitative result of AttGAN.

In [ ]:
from src.utils import attribute_demo
attribute_demo(enc, gen, test_loader, cfg, n_imgs=4)

## Cell 8 — Quantitative evaluation

In [ ]:
from src.utils import evaluate_attribute_accuracy, evaluate_reconstruction

acc = evaluate_attribute_accuracy(enc, gen, dis, test_loader, cfg)
rec = evaluate_reconstruction(enc, gen, test_loader, cfg)

print(f'\nSummary')
print(f'  Attribute accuracy (avg) : {acc:.1f}%')
print(f'  Reconstruction L1        : {rec:.4f}')

---
## Troubleshooting — CelebA download fails

If `torchvision` fails to download CelebA (Google Drive quota error), run this cell instead:

In [ ]:
# Alternative download via Kaggle API
# 1. Upload your kaggle.json API token to Colab Files first
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip install -q kaggle
# !kaggle datasets download -d jessicali9530/celeba-dataset -p data/
# !unzip -q data/celeba-dataset.zip -d data/

# Alternative: mount Google Drive with a pre-downloaded copy
# from google.colab import drive
# drive.mount('/content/drive')
# Then update cfg.DATA_DIR:
# from pathlib import Path
# cfg.DATA_DIR = Path('/content/drive/MyDrive/celeba')
print('Uncomment the method you want to use above.')